In [14]:
from scipy.signal import decimate
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer
import os
import random
import h5py

In [15]:
def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (filename_path , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [16]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [17]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels

In [18]:
import re

def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")


In [19]:
def load_and_preprocess(filepath, label_map, num_chunks=50, downsample_factor=20):
    task_label = infer_label_from_filename(filepath, label_map)

    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # Shape: (248, T)

    # Downsample (along time axis)
    data = decimate(data, q=downsample_factor, axis=1)

    total_length = data.shape[1]
    chunk_length = total_length // num_chunks

    segments = []
    labels = []

    for i in range(num_chunks):
        start = i * chunk_length
        end = start + chunk_length
        if end > total_length:
            break  # Skip last chunk if not full

        window = data[:, start:end]

        # z-score normalization per window (per channel)
        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # (248, chunk_len, 1)
        labels.append(task_label)

    return segments, labels

In [20]:
batch_size = 16

# Set filepaths and label map
data_dir='./data/Intra/train'
filepaths = [
    os.path.normpath(os.path.join(data_dir, fname))
    for fname in os.listdir(data_dir)
    if fname.endswith('.h5')
]
print(filepaths)
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}

['data\\Intra\\train\\rest_105923_1.h5', 'data\\Intra\\train\\rest_105923_2.h5', 'data\\Intra\\train\\rest_105923_3.h5', 'data\\Intra\\train\\rest_105923_4.h5', 'data\\Intra\\train\\rest_105923_5.h5', 'data\\Intra\\train\\rest_105923_6.h5', 'data\\Intra\\train\\rest_105923_7.h5', 'data\\Intra\\train\\rest_105923_8.h5', 'data\\Intra\\train\\task_motor_105923_1.h5', 'data\\Intra\\train\\task_motor_105923_2.h5', 'data\\Intra\\train\\task_motor_105923_3.h5', 'data\\Intra\\train\\task_motor_105923_4.h5', 'data\\Intra\\train\\task_motor_105923_5.h5', 'data\\Intra\\train\\task_motor_105923_6.h5', 'data\\Intra\\train\\task_motor_105923_7.h5', 'data\\Intra\\train\\task_motor_105923_8.h5', 'data\\Intra\\train\\task_story_math_105923_1.h5', 'data\\Intra\\train\\task_story_math_105923_2.h5', 'data\\Intra\\train\\task_story_math_105923_3.h5', 'data\\Intra\\train\\task_story_math_105923_4.h5', 'data\\Intra\\train\\task_story_math_105923_5.h5', 'data\\Intra\\train\\task_story_math_105923_6.h5', 'data

In [21]:
def data_generator(filepaths, label_map, batch_size):
    while True:
        random.shuffle(filepaths)  # Shuffle file order each epoch
        all_segments, all_labels = [], []

        for filepath in filepaths:
            segments, labels = load_and_preprocess(filepath, label_map)
            all_segments.extend(segments)
            all_labels.extend(labels)

        # Convert to numpy arrays
        X = np.array(all_segments)
        y = np.array(all_labels)

        # Shuffle all data once per epoch
        indices = np.random.permutation(len(y))
        X = X[indices]
        y = y[indices]

        # Yield mini-batches
        for i in range(0, len(X), batch_size):
            yield X[i:i + batch_size], y[i:i + batch_size]

In [22]:
segments, _ = load_and_preprocess(filepaths[0], label_map)
print(segments[0].shape)

(248, 35, 1)


In [23]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
def build_cnn(input_shape=segments[0].shape, num_classes=4, l2_lambda=0.001): # shape = (248, 35, 1)
    model = Sequential([
        InputLayer(input_shape=input_shape),

        Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),
 

        Conv2D(64, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Conv2D(128, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Flatten(),
        Dropout(0.6),
        Dense(128, activation='relu', kernel_regularizer=l2(l2_lambda)),
        # Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
   
    model.compile(optimizer=Adam(learning_rate=0.0005),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [24]:
def count_total_segments(filepaths, label_map):
    total = 0
    for fp in filepaths:
        segments, _ = load_and_preprocess(fp, label_map)
        total += len(segments)
    return total

In [25]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(filepaths, test_size=0.2, random_state=42)


In [26]:
from tensorflow.keras.callbacks import EarlyStopping

# Build model
model = build_cnn()

# determine steps per epoch
train_total_segments = count_total_segments(train_files, label_map)
train_steps_per_epoch = train_total_segments // batch_size
train_gen = data_generator(train_files, label_map, batch_size=batch_size)

val_total_segments = count_total_segments(val_files, label_map)
val_steps_per_epoch = val_total_segments // batch_size
val_gen = data_generator(val_files, label_map, batch_size=batch_size)

early_stop = EarlyStopping(
    monitor='val_loss',       
    patience=3,               
    restore_best_weights=True
)

# Fit model
model.fit(
    train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=val_gen,
    validation_steps=val_steps_per_epoch
    # callbacks=[early_stop]
)

Epoch 1/15
78/78 [==============================] - 61s 710ms/step - loss: 1.7731 - accuracy: 0.3013 - val_loss: 1.7211 - val_accuracy: 0.1458
Epoch 2/15
78/78 [==============================] - 35s 448ms/step - loss: 1.5840 - accuracy: 0.3922 - val_loss: 1.6782 - val_accuracy: 0.1488
Epoch 3/15
78/78 [==============================] - 32s 411ms/step - loss: 1.4562 - accuracy: 0.5049 - val_loss: 1.7115 - val_accuracy: 0.1399
Epoch 4/15
78/78 [==============================] - 33s 428ms/step - loss: 1.2445 - accuracy: 0.6515 - val_loss: 1.9718 - val_accuracy: 0.1429
Epoch 5/15
78/78 [==============================] - 34s 433ms/step - loss: 1.1411 - accuracy: 0.7690 - val_loss: 1.8713 - val_accuracy: 0.1429
Epoch 6/15
78/78 [==============================] - 33s 427ms/step - loss: 0.9664 - accuracy: 0.8582 - val_loss: 1.9717 - val_accuracy: 0.1429
Epoch 7/15
78/78 [==============================] - 32s 417ms/step - loss: 0.8540 - accuracy: 0.9133 - val_loss: 2.2554 - val_accuracy: 0.1310

In [29]:
def load_test_data(test_filepaths, label_map, num_chunks=50, downsample_factor=20):
    all_segments = []
    all_labels = []

    for filepath in test_filepaths:
        segments, labels = load_and_preprocess(
            filepath, label_map,
            num_chunks=num_chunks,
            downsample_factor=downsample_factor
        )
        all_segments.extend(segments)
        all_labels.extend(labels)

    X_test = np.array(all_segments)
    y_test = np.array(all_labels)
    return X_test, y_test


In [30]:
import glob

# Collect test files
test_folder = './data/Intra/test'
test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
test_filepaths = [os.path.normpath(p) for p in test_filepaths]

# Load and preprocess test data
X_test, y_test = load_test_data(test_filepaths, label_map)
# Direct evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Accuracy: {accuracy:.4f}")

13/13 [==============================] - 1s 101ms/step - loss: 11.5745 - accuracy: 0.3700
Test Accuracy: 0.3700
